<a href="https://colab.research.google.com/github/liamcohenn/tp_laboratorio_futbol/blob/main/Copia_de_tp_labo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Trabajo Práctico Grupal - Laboratorio**

**Integrantes:** Liam Cohen, Renzo Bianco Maura, Federico Kurozwan, Milagros Leon

**Pregunta:** ¿Qué peso y qué rentabilidad tienen las ventas de jugadores formados en inferiores frente a las de jugadores comprados?

**Dataset:** Football Data from Transfermarkt (tablas transfers, players, clubs y competitions), subido al repositorio del grupo.

**Ligas analizadas:** Premier League (GB1), LaLiga (ES1), Ligue 1 (FR1), Primeira Liga (PO1), Eredivisie (NL1), Argentina (ARG1), México (MEX1) y Arabia Saudita (SA1).

# 3.1. Carga y exploración inicial

 ### Carga de datos



In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

BASE = "https://raw.githubusercontent.com/liamcohenn/tp_laboratorio_futbol/main/datos/raw/"

transfers = pd.read_csv(BASE + "transfers.csv.gz")
players = pd.read_csv(BASE + "players.csv.gz")
clubs = pd.read_csv(BASE + "clubs.csv.gz")
competitions = pd.read_csv(BASE + "competitions.csv.gz")

### Primeras filas de `transfers`

In [3]:
transfers.head()

,player_id,transfer_date,transfer_season,from_club_id,to_club_id,from_club_name,to_club_name,transfer_fee,market_value_in_eur,player_name
0,467994,2030-06-30,25/26,5621,749,Reggiana,FC Empoli,0.0,700000.0,Luca Belardinelli
1,645842,2028-02-02,27/28,6505,19684,Gimcheon Sangmu,Jeju SK,0.0,150000.0,Chan-gi An
2,677470,2028-02-02,27/28,6505,3535,Gimcheon Sangmu,Ulsan HD,0.0,400000.0,Yool Heo
3,709155,2028-02-02,27/28,6505,19684,Gimcheon Sangmu,Jeju SK,0.0,325000.0,Jae-hyeok Oh
4,728133,2028-02-02,27/28,6505,35759,Gimcheon Sangmu,Bucheon FC,0.0,175000.0,Sang-hyeok Lee


### Últimas filas de `transfers`

In [ ]:
transfers.tail()

### Cantidad de filas y columnas de `transfers`:

In [ ]:
transfers.shape

### Tipo de dato de cada columna de `transfers`:

In [ ]:
transfers.dtypes

### Valores nulos en `transfers`:

In [ ]:
transfers.isnull().sum()

### Filas duplicadas en `transfers`:

In [ ]:
transfers.duplicated().sum()

### Estadísticas descriptivas de `transfers`:

In [ ]:
transfers.describe()

#### Pasamos ahora con la tabla `players`
### Primeras filas de `players`:

In [ ]:
players.head()

### Últimas filas de `players`:

In [ ]:
players.tail()

### Cantidad de filas y columnas de `players`:

In [ ]:
players.shape

### Tipo de dato de cada columna de `players`:

In [ ]:
players.dtypes

### Valores nulos en `players`:

In [ ]:
players.isnull().sum()

### Filas duplicadas en `players`:

In [ ]:
players.duplicated().sum()

### Estadísticas descriptivas de `players`:

In [ ]:
players.describe()

## Comentarios sobre la primera parte del analisis


>

- **Tamaño:** `transfers` tiene 175.165 filas y 10 columnas; `players` tiene 50.149 filas y 26 columnas.


- **Duplicados:** no hay filas duplicadas ni en `transfers` ni en `players` (0 en las dos).

- **Fechas:** las transferencias van del 1 de julio de 1993 al 30 de junio de 2030. Hay 520 posteriores al 6 de julio de 2026, que son fechas futuras y las sacamos. Después de filtrar quedan 174.645 filas y la fecha más reciente es el 2 de julio de 2026.

 - **Valores nulos:** lo que más se nota es que a `transfers` le faltan muchos datos de dinero: `transfer_fee` tiene 61.526 nulos y `market_value_in_eur` tiene 68.160. Como no sabemos si esas transferencias fueron gratis o simplemente no se cargó el monto, para ingresos por ventas solo vamos a usar las filas que tienen `transfer_fee`. En `players` los nulos están sobre todo en datos de selección: `current_national_team_id` (46.889), `international_caps` e `international_goals` (30.793 cada una). También falta `agent_name` en 23.296 jugadores.

- **Valores llamativos:** en `transfer_fee` el máximo es de €222 millones, pero la mediana es €0, osea que la mayoría de los movimientos tiene un monto de cero, solo unos pocos son muy caros y no hay valores negativos. El valor de mercado más alto es €180 millones. En `players`, un jugador figura con 17 cm de altura, que no puede ser real y seguramente es un error de carga; la altura máxima es 210 cm.







## 3.2 Pregunta de investigación

**a) ¿Cuál es la pregunta?**
¿Qué peso y qué rentabilidad tienen las ventas de jugadores formados en inferiores frente a las de jugadores comprados, en los clubes de 8 ligas?

**b) ¿Por qué vale la pena responderla?**
Se entiende que formar jugadores en inferiores le sale mucho más barato a un club que comprarlos, ya que después los puede vender y ganar plata con eso. Saber cuánto pesan esas ventas sobre el total, y si dejan más ganancia que la venta de jugadores comprados, ayuda a entender si a un club le conviene invertir en sus inferiores o comprar jugadores ya formados.

**c) ¿Qué respuesta encontraron?**
completar al terminar el trabajo

#3.3. Transformaciones y resumen estadístico

#### **Transformación 1: pasar las fechas a formato fecha.** Las columnas `transfer_date` y `date_of_birth` están como texto. Las convertimos con `pd.to_datetime()` para poder restarlas y compararlas.

In [4]:
transfers["transfer_date"] = pd.to_datetime(transfers["transfer_date"])
players["date_of_birth"] = pd.to_datetime(players["date_of_birth"])

transfers.dtypes

,0
player_id,int64
transfer_date,datetime64[ns]
transfer_season,object
from_club_id,int64
to_club_id,int64
from_club_name,object
to_club_name,object
transfer_fee,float64
market_value_in_eur,float64
player_name,object


#### **Transformación 2: sacar las transferencias posteriores al 6 de julio de 2026.** Son finales de préstamo programados, todavía no pasaron, así que no sirven para el análisis. Primero contamos cuántas hay (deberían ser 520) y después las eliminamos.

In [5]:
fecha_corte = pd.to_datetime("2026-07-06")

print("Transferencias posteriores al corte:", (transfers["transfer_date"] > fecha_corte).sum())

transfers = transfers[transfers["transfer_date"] <= fecha_corte]

print("Fecha máxima ahora:", transfers["transfer_date"].max())
print("Filas que quedan:", transfers.shape[0])

Transferencias posteriores al corte: 520
Fecha máxima ahora: 2026-07-02 00:00:00
Filas que quedan: 174645


#### **Transformación 3: calcular la edad del jugador en cada transferencia.** Pegamos la fecha de nacimiento de cada jugador a la tabla `transfers` (con `merge`, usando `player_id`) y calculamos la edad como la diferencia entre la fecha de la transferencia y la de nacimiento, en años. La vamos a necesitar en la Parte 2 para saber si un jugador entró al club antes de los 18.

In [6]:
transfers = transfers.merge(players[["player_id", "date_of_birth"]], on="player_id", how="left")

transfers["edad"] = (transfers["transfer_date"] - transfers["date_of_birth"]).dt.days / 365.25

transfers[["player_name", "transfer_date", "date_of_birth", "edad"]].head()

,player_name,transfer_date,date_of_birth,edad
0,César Augusto,2026-07-02,2003-08-23,22.858316
1,Kingstone Mutandwa,2026-07-02,2003-01-05,23.488022
2,James Milner,2026-07-01,1986-01-04,40.487337
3,Lukas Podolski,2026-07-01,1985-06-04,41.073238
4,Raúl Albiol,2026-07-01,1985-09-04,40.821355


#### **Transformación 4: lista de clubes de las 8 ligas.** Armamos la lista de ligas y nos quedamos con los clubes de la tabla `clubs` que juegan en alguna de ellas.

In [7]:
ligas = ["GB1", "ES1", "FR1", "PO1", "NL1", "ARG1", "MEX1", "SA1"]

clubes_ligas = clubs[clubs["domestic_competition_id"].isin(ligas)]

clubes_ligas["domestic_competition_id"].value_counts()

,count
domestic_competition_id,
GB1,37
FR1,36
PO1,35
ES1,33
ARG1,32
NL1,29
SA1,21
MEX1,18


Transformación 5: nombres de los clubes de las 8 ligas. La tabla clubs tiene los nombres oficiales largos (por ejemplo, “Futbol Club Barcelona”), pero en transfers aparecen los nombres cortos (“Barcelona”). Armamos un diccionario con el nombre corto de cada club, que vamos a usar para reconocer a sus equipos juveniles.

Asistencia de IA: las celdas de la Parte 2 se desarrollaron con asistencia de Claude. Prompt utilizado:[“Escribí en pandas una regla para clasificar las ventas de jugadores como canteranos o comprados, unificando los equipos juveniles con su club y sin contar los regresos de préstamo como compras”].

In [8]:
import re

ids_ligas = set(clubes_ligas["club_id"])

salidas = transfers[transfers["from_club_id"].isin(ids_ligas)][["from_club_id", "from_club_name"]]
salidas.columns = ["club_id", "nombre"]
llegadas = transfers[transfers["to_club_id"].isin(ids_ligas)][["to_club_id", "to_club_name"]]
llegadas.columns = ["club_id", "nombre"]

nombre_club = pd.concat([salidas, llegadas]).groupby("club_id")["nombre"].agg(lambda s: s.value_counts().index[0])
print("Clubes de las 8 ligas:", len(nombre_club))

Clubes de las 8 ligas: 241


Transformación 6: reconocer los equipos juveniles. Transfermarkt registra a los equipos juveniles y filiales como clubes distintos (“Ajax U21”, “Jong Ajax”, “Sporting CP U23”). Esta función le saca al nombre las marcas de juvenil (U21, B, II, Youth, etc.) y las siglas como FC o CF, y avisa si el nombre tenía alguna marca de juvenil. Así, “Sporting CP U23” queda como “sporting”, igual que el club de primera.

In [9]:
JUVENIL = r"\s+(U\d{2}|Y\d{2}|Sub-?\d{2}|Youth|Yout|You|Yth\.?|Y|Jgd\.?|Juv\.?|Juvenil|Res\.?|Reserves?|B|C|II|III|Atlètic|23)$"
SIGLAS = {"fc", "cf", "ud", "rcd", "as", "ca", "ea", "sc", "ac", "afc", "sl", "cp", "club"}

def normalizar(nombre):
    original = str(nombre).strip()
    n = re.sub(r"^Jong\s+", "", original)
    es_juvenil = n != original
    while re.search(JUVENIL, n):
        n = re.sub(JUVENIL, "", n).strip()
        es_juvenil = True
    palabras = [p for p in n.lower().split() if p not in SIGLAS]
    return " ".join(palabras), es_juvenil

print(normalizar("Ajax U21"))
print(normalizar("Jong Ajax"))
print(normalizar("Sporting CP U23"))
print(normalizar("River Plate"))

('ajax', True)
('ajax', True)
('sporting', True)
('river plate', False)


Transformación 7: unificar los equipos juveniles con su club. A cada club de cada transferencia le asignamos su “familia”: si es un juvenil de un club de las 8 ligas, la familia es ese club; si no, es el mismo club. Solo unificamos cuando el nombre tenía una marca de juvenil, para no confundir clubes distintos con el mismo nombre (por ejemplo, River Plate de Uruguay). Dos juveniles no siguen el patrón y se agregan a mano: “Barça Atlètic” y “CABJ U20”.

In [10]:
id_por_nombre_normalizado = {normalizar(nombre)[0]: club_id for club_id, nombre in nombre_club.items()}
id_por_nombre = {nombre: club_id for club_id, nombre in nombre_club.items()}

EQUIVALENCIAS_MANUALES = {"Barça Atlètic": "Barcelona", "CABJ U20": "Boca Juniors"}

def club_familia(club_id, nombre):
    if club_id in ids_ligas:
        return club_id
    if nombre in EQUIVALENCIAS_MANUALES:
        return id_por_nombre[EQUIVALENCIAS_MANUALES[nombre]]
    nombre_norm, es_juvenil = normalizar(nombre)
    if es_juvenil and nombre_norm in id_por_nombre_normalizado:
        return id_por_nombre_normalizado[nombre_norm]
    return club_id

pares = pd.concat([
    transfers[["from_club_id", "from_club_name"]].set_axis(["club_id", "nombre"], axis=1),
    transfers[["to_club_id", "to_club_name"]].set_axis(["club_id", "nombre"], axis=1),
]).drop_duplicates()
pares["familia"] = [club_familia(i, n) for i, n in zip(pares["club_id"], pares["nombre"])]
mapa_familia = dict(zip(zip(pares["club_id"], pares["nombre"]), pares["familia"]))

transfers["familia_origen"] = [mapa_familia[(i, n)] for i, n in zip(transfers["from_club_id"], transfers["from_club_name"])]
transfers["familia_destino"] = [mapa_familia[(i, n)] for i, n in zip(transfers["to_club_id"], transfers["to_club_name"])]

print("Equipos juveniles unificados con su club:", (pares["familia"] != pares["club_id"]).sum())

Equipos juveniles unificados con su club: 748


Transformación 8: separar los pases internos y quedarnos con las ventas. Un pase del juvenil a la primera del mismo club (por ejemplo, de “Benfica B” a “Benfica”) no es una venta, así que lo separamos. Después nos quedamos con las ventas de clubes de las 8 ligas que tienen monto mayor a cero, porque los préstamos y pases libres no generan ingresos.

In [11]:
internos = transfers[transfers["familia_origen"] == transfers["familia_destino"]]
movimientos = transfers[transfers["familia_origen"] != transfers["familia_destino"]].sort_values(["player_id", "transfer_date"])

ventas_base = movimientos[movimientos["familia_origen"].isin(ids_ligas) & (movimientos["transfer_fee"] > 0)]

print("Movimientos internos (juveniles a primera):", len(internos))
print("Ventas con monto de clubes de las 8 ligas:", len(ventas_base))

Movimientos internos (juveniles a primera): 8855
Ventas con monto de clubes de las 8 ligas: 5280


Transformación 9: clasificar cada venta como cantera o comprado. Para cada venta buscamos cómo llegó el jugador al club, recorriendo sus llegadas de la más reciente a la más vieja:
Si la llegada es una vuelta de préstamo (llegó sin monto desde un club al que antes había salido sin monto), la ignoramos y seguimos buscando.
Si llegó antes de los 18 años y sin que se pagara nada, es cantera.
Si llegó después de los 18 o se pagó por él, es comprado, y el costo es lo que se pagó.
Si no tiene ninguna llegada registrada, estuvo en el club desde las inferiores: es cantera.

Así se corrigen los errores de la primera versión: un canterano que sale a préstamo y vuelve ya no cuenta como comprado gratis, y un juvenil comprado (como Frenkie de Jong, por quien el Ajax pagó €7,74 M para su sub-21) ya no cuenta como canterano.

In [12]:
movimientos_por_jugador = dict(tuple(movimientos.groupby("player_id")))

primera_fecha_en_club = pd.concat([
    transfers[["player_id", "familia_origen", "transfer_date"]].set_axis(["player_id", "club", "fecha"], axis=1),
    transfers[["player_id", "familia_destino", "transfer_date"]].set_axis(["player_id", "club", "fecha"], axis=1),
]).groupby(["player_id", "club"])["fecha"].min()

def sin_monto(monto):
    return pd.isna(monto) or monto == 0

def clasificar(venta):
    club = venta.familia_origen
    movs = movimientos_por_jugador[venta.player_id]
    anteriores = movs[movs["transfer_date"] < venta.transfer_date]
    salidas_club = anteriores[anteriores["familia_origen"] == club]
    llegadas_club = anteriores[anteriores["familia_destino"] == club].sort_values("transfer_date", ascending=False)

    for _, llegada in llegadas_club.iterrows():
        salio_prestado = salidas_club[
            (salidas_club["familia_destino"] == llegada["familia_origen"])
            & (salidas_club["transfer_date"] < llegada["transfer_date"])
            & salidas_club["transfer_fee"].apply(sin_monto)
        ]
        if sin_monto(llegada["transfer_fee"]) and len(salio_prestado) > 0:
            continue
        if llegada["edad"] < 18 and sin_monto(llegada["transfer_fee"]):
            return "cantera", 0.0, llegada["transfer_date"]
        return "comprado", llegada["transfer_fee"], llegada["transfer_date"]

    return "cantera", 0.0, primera_fecha_en_club[(venta.player_id, club)]

resultados = [clasificar(v) for v in ventas_base.itertuples()]
ventas_base = ventas_base.assign(
    grupo=[r[0] for r in resultados],
    costo=[r[1] for r in resultados],
    fecha_llegada=[r[2] for r in resultados],
)
ventas_base["grupo"].value_counts()

,count
grupo,
comprado,3911
cantera,1369


Tabla final de ventas. Armamos la tabla ventas con una fila por venta y las columnas que usan los resúmenes, los gráficos y el sitio web. Los montos se pasan a millones de euros para que sean más fáciles de leer.

In [13]:
liga_por_club = dict(zip(clubs["club_id"], clubs["domestic_competition_id"]))

ventas = pd.DataFrame({
    "jugador": ventas_base["player_name"],
    "club_vendedor": ventas_base["familia_origen"].map(nombre_club),
    "club_comprador": ventas_base["to_club_name"],
    "liga": ventas_base["familia_origen"].map(liga_por_club),
    "temporada_venta": ventas_base["transfer_date"].dt.year,
    "grupo": ventas_base["grupo"],
    "fee_venta": ventas_base["transfer_fee"] / 1_000_000,
    "fee_compra": ventas_base["costo"] / 1_000_000,
    "anios_en_club": ((ventas_base["transfer_date"] - ventas_base["fecha_llegada"]).dt.days / 365.25).round(1),
    "edad_venta": ventas_base["edad"].round(1),
}).reset_index(drop=True)
ventas.head()

,jugador,club_vendedor,club_comprador,liga,temporada_venta,grupo,fee_venta,fee_compra,anios_en_club,edad_venta
0,James Milner,Leeds,Newcastle,GB1,2004,cantera,7.40,0.0,2.0,18.5
1,James Milner,Newcastle,Aston Villa,GB1,2008,comprado,15.00,7.4,4.2,22.7
2,James Milner,Aston Villa,Man City,GB1,2010,comprado,22.00,15.0,2.0,24.6
3,Boy Waterman,Heerenveen,AZ Alkmaar,NL1,2007,cantera,3.00,0.0,6.0,23.4
4,Pepe Reina,Barcelona,Villarreal,ES1,2002,cantera,0.75,0.0,3.0,19.8


Control de la clasificación. Verificamos la regla con casos conocidos: Gonçalo Ramos y Rúben Dias se formaron en Benfica; Frenkie de Jong fue comprado por el Ajax para su sub-21; Enzo Fernández se formó en River y Benfica lo compró. También contamos cuántos comprados tienen costo 0, que deberían ser solo los que llegaron con el pase libre.

In [14]:
casos = [("Gonçalo Ramos", "Benfica", "cantera"),
         ("Rúben Dias", "Benfica", "cantera"),
         ("Frenkie de Jong", "Ajax", "comprado"),
         ("Enzo Fernández", "River Plate", "cantera"),
         ("Enzo Fernández", "Benfica", "comprado")]

for jugador, club, esperado in casos:
    fila = ventas[(ventas["jugador"] == jugador) & (ventas["club_vendedor"] == club)]
    obtenido = fila["grupo"].iloc[0] if len(fila) else "NO ENCONTRADO"
    print("OK   " if obtenido == esperado else "ERROR", jugador, "-", club, "→", obtenido)

comprados = ventas[ventas["grupo"] == "comprado"]
print("\nVentas totales:", len(ventas))
print("Comprados con costo 0 (pases libres):", (comprados["fee_compra"] == 0).sum(), "de", len(comprados))
print("Comprados con costo desconocido:", comprados["fee_compra"].isna().sum())
print(ventas.groupby("liga")["grupo"].value_counts().unstack())

OK    Gonçalo Ramos - Benfica → cantera
OK    Rúben Dias - Benfica → cantera
OK    Frenkie de Jong - Ajax → comprado
OK    Enzo Fernández - River Plate → cantera
OK    Enzo Fernández - Benfica → comprado

Ventas totales: 5280
Comprados con costo 0 (pases libres): 680 de 3911
Comprados con costo desconocido: 725
grupo  cantera  comprado
liga                    
ARG1       251       378
ES1        154       660
FR1        264       788
GB1        283       857
MEX1        39       176
NL1        205       451
PO1        131       542
SA1         42        59
